# NiyamTrace-X Q1 Experiment 06 — Metamorphic Multilingual Semantic Attack Lab

**Purpose.** Replace the narrow first-wave Anchor stress test with a differential, metamorphic robustness laboratory built from the exact frozen holdout.

The experiment compares:

- **Frozen Anchor Lock logic** reproduced from the frozen runtime.
- **Hardened Anchor Lock v2**, which adds Unicode-format stripping, a small confusable-character normalization layer, contextual year extraction that excludes vendor-ID spans, and broader punctuation/zero-width tolerance.

### Attack families
- clean/equivalent transformations (must not create a violation),
- entity, time, month, negation and quantifier drift,
- Unicode/full-width digits,
- zero-width insertion,
- confusable characters in the word *vendor*,
- punctuation/whitespace perturbations,
- compound multi-field corruption,
- the discovered **vendor-ID/year collision regression**.

This is a controlled security stress test, not a substitute for the frozen empirical LLM evaluation. It reports both **attack recall** and **clean false-positive rate**, plus automatically saved failure examples.


In [ ]:
import importlib.util, subprocess, sys
need=[p for p in ['pandas','numpy','matplotlib'] if importlib.util.find_spec(p) is None]
if need: subprocess.check_call([sys.executable,'-m','pip','install','-q']+need)


In [ ]:
from pathlib import Path
import os, json, math, hashlib, zipfile, shutil, random, statistics, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20260911
random.seed(SEED)
np.random.seed(SEED)

BASE = Path('/content') if Path('/content').exists() else Path('/mnt/data')
RESULTS = BASE / 'niyamtrace_q1_wave2_results'
RESULTS.mkdir(parents=True, exist_ok=True)
EVIDENCE_DIR = BASE / 'ntx_frozen_evidence'
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED = {
    'holdout2000.jsonl': '4dad5dad9ea4a3258f6139407664ab2584678b440622871fa1b5d5da24f95d3a',
    'qwen_results.jsonl': '7d1b3bf2b0b34d0101c1a885847d6d511a54014c16f43e4cea778b6078123eb7',
    'gptoss_results.jsonl': '991812849fb9f5a0651b6277b7bc71d6c2c1ca8796b166eb182ec8fac221e81d',
}

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def find_or_upload_evidence():
    candidates = [
        BASE/'NiyamTrace-X_Frozen_Raw_Evidence_Recovered.zip',
        Path.cwd()/'NiyamTrace-X_Frozen_Raw_Evidence_Recovered.zip',
    ]
    for p in candidates:
        if p.exists(): return p
    try:
        from google.colab import files
        print('Upload NiyamTrace-X_Frozen_Raw_Evidence_Recovered.zip')
        uploaded = files.upload()
        for name, data in uploaded.items():
            p=BASE/name
            p.write_bytes(data)
            if name.endswith('.zip'): return p
    except Exception as e:
        raise FileNotFoundError('Place NiyamTrace-X_Frozen_Raw_Evidence_Recovered.zip in /content or current directory.') from e
    raise FileNotFoundError('Evidence ZIP not found.')

ZIP = find_or_upload_evidence()
with zipfile.ZipFile(ZIP) as z:
    z.extractall(EVIDENCE_DIR)

# Accept either flat package or a nested folder.
def locate(name):
    hits=list(EVIDENCE_DIR.rglob(name))
    if len(hits)!=1:
        raise RuntimeError(f'Expected exactly one {name}, found {len(hits)}: {hits[:5]}')
    return hits[0]

paths={k:locate(k) for k in EXPECTED}
for name, exp in EXPECTED.items():
    got=sha256(paths[name])
    print(name, got, 'OK' if got==exp else 'HASH MISMATCH')
    assert got==exp, (name, got, exp)

holdout=pd.read_json(paths['holdout2000.jsonl'], lines=True)
qwen=pd.read_json(paths['qwen_results.jsonl'], lines=True)
gpt=pd.read_json(paths['gptoss_results.jsonl'], lines=True)
assert len(holdout)==len(qwen)==len(gpt)==2000
assert holdout.variant_group_id.nunique()==500
assert set(qwen.case_id)==set(holdout.case_id)==set(gpt.case_id)
print('Evidence verified:', len(holdout), 'cases /', holdout.variant_group_id.nunique(), 'semantic groups')


In [ ]:
import unicodedata, copy
from dataclasses import dataclass

MONTHS={
 'january':1,'jan':1,'february':2,'feb':2,'march':3,'mar':3,'april':4,'apr':4,'may':5,'june':6,'jun':6,
 'july':7,'jul':7,'august':8,'aug':8,'september':9,'sep':9,'sept':9,'october':10,'oct':10,'november':11,'nov':11,'december':12,'dec':12,
}
NEG=[r'\bdo\s+not\b',r"\bdon['’]?t\b",r'\bnot\b',r'\bnever\b',r'\bmat\b',r'\bnahi\b',r'\bnahin\b',r'\bcheyyakandi\b',r'\bcheyya?kandi\b',r'\bcheyyavaddu\b',r'\bcheyavaddu\b',r'\bvaddu\b',r'చేయకండి',r'చేయవద్దు',r'వద్దు']
BROAD=[r'\ball\b',r'\bevery\b',r'\bsabhi\b',r'\bsab\b',r'\bsare\b',r'\banni\b',r'\bandarini\b',r'అన్ని',r'అన్నీ']
NARROW=[r'\bonly\b',r'\bsirf\b',r'\bkeval\b',r'\bmaatrame\b',r'\bmatrame\b',r'మాత్రమే']
NARROW_HARD=NARROW+[r'\bmatching\b']
VENDOR_RE=re.compile(r'(?:\bvendor\b|వెండర్)\s*[-:#]?\s*(\d{2,})',re.I)
YEAR_RE=re.compile(r'\b(20\d{2})\b')

CONFUSABLES=str.maketrans({'е':'e','Е':'E','о':'o','О':'O','а':'a','А':'A','р':'p','Р':'P','с':'c','С':'C','х':'x','Х':'X','ν':'v'})

def norm_frozen(text): return unicodedata.normalize('NFKC',text).casefold()
def norm_hardened(text):
    x=unicodedata.normalize('NFKC',text).translate(CONFUSABLES)
    x=''.join(ch for ch in x if unicodedata.category(ch)!='Cf')
    return x.casefold()
def anyp(t,ps): return any(re.search(p,t,re.I) for p in ps)

def frozen_anchors(text):
    t=norm_frozen(text)
    vendors=list(dict.fromkeys(VENDOR_RE.findall(t)))
    years=list(dict.fromkeys(int(x) for x in YEAR_RE.findall(t)))
    months=[]
    for n,m in MONTHS.items():
        if re.search(rf'\b{re.escape(n)}\b',t): months.append(m)
    return {'negated':anyp(t,NEG),'broad':anyp(t,BROAD),'narrow':anyp(t,NARROW),'vendors':vendors,'years':list(dict.fromkeys(years)),'months':list(dict.fromkeys(months))}

def hardened_anchors(text):
    t=norm_hardened(text)
    vendor_matches=list(re.finditer(r'(?:\bvendor\b|వెండర్)\s*[-:#]?\s*(\d{2,})',t,re.I))
    vendors=list(dict.fromkeys(m.group(1) for m in vendor_matches))
    vendor_spans=[m.span(1) for m in vendor_matches]
    # Context-aware year extraction: ignore 20xx tokens that are the vendor identifier itself.
    years=[]
    for m in re.finditer(r'\b(20\d{2})\b',t):
        if any(a<=m.start(1) and m.end(1)<=b for a,b in vendor_spans):
            continue
        years.append(int(m.group(1)))
    months=[]
    for n,m in MONTHS.items():
        if re.search(rf'\b{re.escape(n)}\b',t): months.append(m)
    return {'negated':anyp(t,NEG),'broad':anyp(t,BROAD),'narrow':anyp(t,NARROW_HARD),'vendors':vendors,'years':list(dict.fromkeys(years)),'months':list(dict.fromkeys(months))}

def canon_scope(x):
    if x is None:return None
    x=str(x).strip().casefold().replace('_',' ')
    return {'only matching':'matching','matching only':'matching'}.get(x,x)

def violations(text,cand,extractor):
    a=extractor(text); bad=set()
    if a['negated'] and not bool(cand.get('negated')): bad.add('negation')
    if a['vendors'] and str(cand.get('vendor_id')) not in set(a['vendors']): bad.add('vendor_id')
    if a['years'] and cand.get('year') not in set(a['years']): bad.add('year')
    if a['months'] and cand.get('month') not in set(a['months']): bad.add('month')
    s=canon_scope(cand.get('scope'))
    if a['broad'] and s!='all': bad.add('quantifier_all')
    if a['narrow'] and not a['broad'] and s not in {'matching','only matching'}: bad.add('quantifier_only')
    return sorted(bad)


In [ ]:
# Metamorphic transforms and candidate corruption operators.
MONTH_NAMES={1:'January',2:'February',3:'March',4:'April',5:'May',6:'June',7:'July',8:'August',9:'September',10:'October',11:'November',12:'December'}
FULLWIDTH=str.maketrans('0123456789','０１２３４５６７８９')

def replace_once(text,old,new):
    return re.sub(re.escape(str(old)),str(new),text,count=1,flags=re.I)

def equivalent_surface_variants(text, contract):
    out=[]
    out.append(('clean',text))
    out.append(('fullwidth_digits',text.translate(FULLWIDTH)))
    out.append(('extra_whitespace',re.sub(r'\s+','   ',text)))
    out.append(('punctuation_noise',text.replace('vendor ','vendor: ')))
    # Zero-width around punctuation/words, not inside the semantic numeric value.
    out.append(('zero_width_format',text.replace('vendor','ven\u200bdor')))
    out.append(('vendor_confusable_label',text.replace('vendor','vеndor').replace('Vendor','Vеndor')))
    return out

def corruptions(contract):
    c=copy.deepcopy(contract); out=[]
    if c.get('vendor_id') is not None:
        x=copy.deepcopy(c); x['vendor_id']=str((int(str(c['vendor_id']))+137)%9000+1000); out.append(('entity_shift',x))
    if c.get('year') is not None:
        x=copy.deepcopy(c); x['year']=int(c['year'])+1; out.append(('year_shift',x))
    if c.get('month') is not None:
        x=copy.deepcopy(c); x['month']=int(c['month'])%12+1; out.append(('month_shift',x))
    if bool(c.get('negated')):
        x=copy.deepcopy(c); x['negated']=False; out.append(('negation_drop',x))
    if canon_scope(c.get('scope'))=='matching':
        x=copy.deepcopy(c); x['scope']='all'; out.append(('quantifier_broaden',x))
    if canon_scope(c.get('scope'))=='all':
        x=copy.deepcopy(c); x['scope']='matching'; out.append(('quantifier_narrow',x))
    # Compound candidate corruption when possible.
    x=copy.deepcopy(c); touched=0
    if x.get('vendor_id') is not None: x['vendor_id']=str((int(str(x['vendor_id']))+311)%9000+1000); touched+=1
    if x.get('year') is not None: x['year']=int(x['year'])+2; touched+=1
    if touched>=2: out.append(('compound_vendor_year',x))
    return out


In [ ]:
# Construct the stress corpus from all frozen cases.
records=[]
for _,r in holdout.iterrows():
    base={k:r[k] for k in ['case_id','variant_group_id','language','relation','risk','raw_text']}
    gold=copy.deepcopy(r.semantic_expected)
    # Equivalent surface transformations: correct candidate should not be rejected.
    for fam,txt in equivalent_surface_variants(r.raw_text,gold):
        records.append({**base,'family':fam,'kind':'clean_equivalent','text':txt,'candidate':gold,'should_trigger':False})
    # Candidate semantic drift against the original surface.
    for fam,cand in corruptions(gold):
        records.append({**base,'family':fam,'kind':'candidate_drift','text':r.raw_text,'candidate':cand,'should_trigger':True})

    # Evasion combinations: obfuscate the vendor label while also drifting the vendor ID.
    # These distinguish surface-normalization robustness from ordinary anchor extraction.
    if gold.get('vendor_id') is not None and re.search(r'\bvendor\b', r.raw_text, flags=re.I):
        bad=copy.deepcopy(gold); bad['vendor_id']=str((int(str(gold['vendor_id']))+557)%9000+1000)
        records.append({**base,'family':'entity_shift_zero_width_label','kind':'candidate_drift','text':re.sub(r'(?i)vendor','ven\u200bdor',r.raw_text),'candidate':bad,'should_trigger':True})
        records.append({**base,'family':'entity_shift_confusable_label','kind':'candidate_drift','text':re.sub(r'(?i)vendor','vеndor',r.raw_text),'candidate':bad,'should_trigger':True})

# Explicit regression cases for the discovered vendor-ID/year collision.
for vendor_id in ['2024','2025','2026','2031','2044','2099']:
    text=f'Archive matching November 2026 invoices for vendor {vendor_id}.'
    cand={'action':'invoice.archive','vendor_id':vendor_id,'month':11,'year':2026,'amount':None,'currency':None,'scope':'matching','negated':False}
    records.append({'case_id':f'regression-{vendor_id}','variant_group_id':f'regression-{vendor_id}','language':'eng_Latn','relation':'EQUIVALENT','risk':'LOW','raw_text':text,'family':'vendor_year_collision','kind':'clean_equivalent','text':text,'candidate':cand,'should_trigger':False})

stress=pd.DataFrame(records)
print('Stress rows:',len(stress),'families:',stress.family.nunique())


In [ ]:
# Differential evaluation.
def eval_detector(row, extractor):
    v=violations(row.text,row.candidate,extractor)
    return bool(v), '|'.join(v)

for name,ex in [('frozen',frozen_anchors),('hardened_v2',hardened_anchors)]:
    vals=stress.apply(lambda r:eval_detector(r,ex),axis=1)
    stress[f'{name}_trigger']=[x[0] for x in vals]
    stress[f'{name}_violations']=[x[1] for x in vals]

rows=[]
for det in ['frozen','hardened_v2']:
    pred=stress[f'{det}_trigger']
    y=stress.should_trigger
    tp=int((pred & y).sum()); fn=int((~pred & y).sum()); fp=int((pred & ~y).sum()); tn=int((~pred & ~y).sum())
    rows.append({'detector':det,'n':len(stress),'tp':tp,'fn':fn,'fp':fp,'tn':tn,'attack_recall':tp/(tp+fn) if tp+fn else np.nan,'clean_fpr':fp/(fp+tn) if fp+tn else np.nan,'accuracy':(tp+tn)/len(stress)})
summary=pd.DataFrame(rows)
display(summary)
summary.to_csv(RESULTS/'exp06_detector_summary.csv',index=False)
stress.to_json(RESULTS/'exp06_metamorphic_cases.jsonl',orient='records',lines=True,force_ascii=False)


In [ ]:
# Per-family results and automatic failure corpus.
fam=[]
for det in ['frozen','hardened_v2']:
    for family,g in stress.groupby('family'):
        pred=g[f'{det}_trigger']; y=g.should_trigger
        fam.append({'detector':det,'family':family,'n':len(g),'expected_trigger_rate':y.mean(),'trigger_rate':pred.mean(),'correct_rate':(pred==y).mean(),'false_positive':int((pred & ~y).sum()),'false_negative':int((~pred & y).sum())})
fam=pd.DataFrame(fam)
fam.to_csv(RESULTS/'exp06_family_metrics.csv',index=False)
display(fam)

fail=stress[(stress.hardened_v2_trigger!=stress.should_trigger)].copy()
print('Hardened failures:',len(fail))
fail.to_json(RESULTS/'exp06_hardened_failures.jsonl',orient='records',lines=True,force_ascii=False)

reg=stress[stress.family=='vendor_year_collision'][['text','candidate','frozen_trigger','frozen_violations','hardened_v2_trigger','hardened_v2_violations']]
print('Vendor/year collision regression:')
display(reg)
reg.to_csv(RESULTS/'exp06_vendor_year_collision_regression.csv',index=False)


In [ ]:
# Property/invariant checks.
checks={
 'no_year_collision_fp_v2': bool((~stress.loc[stress.family=='vendor_year_collision','hardened_v2_trigger']).all()),
 'all_clean_equivalent_v2': bool((~stress.loc[stress.kind=='clean_equivalent','hardened_v2_trigger']).all()),
 'candidate_drift_recall_v2': float(stress.loc[stress.kind=='candidate_drift','hardened_v2_trigger'].mean()),
}
print(json.dumps(checks,indent=2))
(RESULTS/'exp06_property_checks.json').write_text(json.dumps(checks,indent=2),encoding='utf-8')
# Do not force a pass threshold here: the experiment is designed to expose residual gaps honestly.


In [ ]:
# Visualization: per-family correct detection rate.
p=fam.pivot(index='family',columns='detector',values='correct_rate').sort_values('hardened_v2')
ax=p.plot(kind='barh',figsize=(8,7))
ax.set_xlabel('Correct classification rate'); ax.set_xlim(0,1.02); ax.set_title('Differential metamorphic robustness by attack family')
plt.tight_layout(); plt.savefig(RESULTS/'exp06_family_robustness.png',dpi=220,bbox_inches='tight'); plt.show()

# Manuscript-ready summary.
latex=summary.copy(); latex['attack_recall']=(latex.attack_recall*100).map(lambda x:f'{x:.2f}\\%'); latex['clean_fpr']=(latex.clean_fpr*100).map(lambda x:f'{x:.2f}\\%')
(RESULTS/'exp06_detector_table.tex').write_text(latex[['detector','n','attack_recall','clean_fpr','fp','fn']].to_latex(index=False,escape=False),encoding='utf-8')


In [ ]:
manifest={'experiment':'NTX_Q1_06_Metamorphic_Semantic_Attack_Lab','seed':SEED,'n_rows':int(len(stress)),'n_families':int(stress.family.nunique()),'input_sha256':EXPECTED,'warning':'Controlled metamorphic stress test; do not describe as natural-distribution LLM accuracy.'}
(RESULTS/'exp06_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')


In [ ]:
# FINAL CELL — package this notebook's complete results and download the ZIP.
from pathlib import Path
import zipfile, hashlib

PREFIX = 'exp06_'
ZIP_OUT = BASE / 'NTX_Q1_06_METAMORPHIC_ATTACK_LAB_RESULTS.zip'
with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS.iterdir()):
        if p.is_file() and p.name.startswith(PREFIX):
            z.write(p, arcname=p.name)

sha = hashlib.sha256(ZIP_OUT.read_bytes()).hexdigest()
print('Created:', ZIP_OUT)
print('SHA-256:', sha)
print('Size MiB:', round(ZIP_OUT.stat().st_size/1024**2, 3))
try:
    from google.colab import files
    files.download(str(ZIP_OUT))
except Exception:
    print('Not running in Colab; ZIP is available at:', ZIP_OUT)
